# Release metro waves

Computes release waves via topological sort of inter-repository dependencies. Wave 0 contains repositories with no internal dependencies that can release first. Each subsequent wave depends only on earlier waves.

In [ ]:
repository_filter: list[str] = []
data_file_dependencies: str = (
    "../samples/v2/org.openrewrite.maven.table.DependenciesDeclared.csv"
)
data_file_parents: str = (
    "../samples/v2/io.moderne.recipe.releasemetro.table.ParentRelationships.csv"
)

In [ ]:
from moderne_visualizations_misc.reusable.data_loader import (
    read_data_table,
    read_optional_csv,
)
from code_data_science import (
    data_table as dt,
    unique_dictionaries as ud,
    tree_data_grid,
)
from moderne_visualizations_misc.reusable.release_metro_utils import (
    build_release_graph,
    compute_release_waves,
)
from moderne_visualizations_misc.reusable.quality_utils import (
    filter_repos,
    short_repo,
)
import warnings

warnings.simplefilter("ignore")

# Load primary table (ProjectCoordinates, overridden by NB_DATA_TABLE).
coords_df = read_data_table(
    "../samples/v2/io.moderne.recipe.releasemetro.table.ProjectCoordinates.csv"
)
# DependenciesDeclared and ParentRelationships are additionalDataTables
# and may be absent: the platform injects None for the parameter when
# the recipe run didn't produce the table (different recipe didn't run,
# Gradle-only project for ParentRelationships, no Java modules, etc.).
deps_df = read_optional_csv(data_file_dependencies)
parents_df = read_optional_csv(data_file_parents)

coords_df = filter_repos(coords_df, repository_filter)
if len(deps_df) > 0:
    deps_df = filter_repos(deps_df, repository_filter)
if len(parents_df) > 0:
    parents_df = filter_repos(parents_df, repository_filter)

In [ ]:
if len(coords_df) > 1 and len(deps_df) == 0:
    # An empty DependenciesDeclared table means the dependency data did not reach
    # the notebook (see release_metro_plan). Skip wave ordering and show an
    # explicit notice rather than a misleading single "Wave 0" that lumps every
    # repository together. A genuinely edge-free portfolio keeps a non-empty deps
    # table and still renders its waves below.
    notice = ud.UniqueDictionaries()
    notice.add(
        {
            "path": "Release Waves/Dependency data unavailable — wave ordering skipped",
            "depends_on": "—",
            "required_by": "—",
        }
    )
    tree_data_grid.display(
        notice.to_list(),
        "Release Waves — dependency data unavailable",
        [
            {"field": "depends_on", "headerName": "Depends on", "minWidth": 120},
            {"field": "required_by", "headerName": "Required by", "minWidth": 120},
        ],
    )
else:
    G = build_release_graph(coords_df, deps_df, parents_df)
    waves, circular = compute_release_waves(G)

    total_modules = len(G.nodes())
    releasable = sum(len(w) for w in waves)
    wave_count = len(waves)

    tree = ud.UniqueDictionaries()

    for i, wave in enumerate(waves):
        wave_label = f"Wave {i} ({len(wave)} modules)"
        for repo in sorted(wave):
            tree.add(
                {
                    "path": f"Release Waves/{wave_label}/{short_repo(repo)}",
                    "depends_on": str(G.in_degree(repo)),
                    "required_by": str(G.out_degree(repo)),
                }
            )

    if circular:
        circ_label = f"Circular ({len(circular)} modules)"
        for repo in circular:
            tree.add(
                {
                    "path": f"Release Waves/{circ_label}/{short_repo(repo)}",
                    "depends_on": str(G.in_degree(repo)),
                    "required_by": str(G.out_degree(repo)),
                }
            )

    tree_data_grid.display(
        tree.to_list(),
        f"Release Waves — {releasable} of {total_modules} modules in {wave_count} waves",
        [
            {"field": "depends_on", "headerName": "Depends on", "minWidth": 120},
            {"field": "required_by", "headerName": "Required by", "minWidth": 120},
        ],
    )
